# The Bodyweight Tax: What Your Joints Are Actually Carrying

Someone posted that they and a lighter friend ran the *same pace, same shoes,
same surface* during marathon training — and only the heavier one wound up with
cooked shins and calves. A lot of people said "that's just being out of shape."

It isn't. The physics are literally different at different bodyweights, and the
gap compounds every single step. So I ported the joint-load model behind the
Yunvo calculator into Python and ran the numbers.

**The model (peak force through one joint, per step):**

```
F = BW × base[activity][joint] × speedMod × cadenceMod × strikeMod
```

Cumulative load over a distance is just `F × steps`. Everything below is that
one equation, swept across bodyweight, pace, cadence, strike, and distance.

Peak-force multiples, activity/speed/cadence/strike coefficients are drawn from
the biomechanics literature — see the README for full sourcing.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager  # noqa: F401  (kept for parity with site font work)

Joint = Literal["ankle", "knee", "hip", "tibia"]
Activity = Literal["walking", "running"]
Strike = Literal["heel", "midfoot", "forefoot"]
WalkPace = Literal["slow", "normal", "brisk"]

# ── Model coefficients (ported 1:1 from the calculator JS) ──────────────────
# Peak force as a multiple of bodyweight, per joint, at reference conditions.
BASE: dict[str, dict[str, float]] = {
    "walking": {"ankle": 1.2, "knee": 2.8, "hip": 2.8, "tibia": 0.9},
    "running": {"ankle": 7.0, "knee": 5.5, "hip": 5.5, "tibia": 3.2},
}

# Foot-strike multipliers. Running-relevant; walking is biomechanically heel-first
# and is not penalised (defaults to midfoot in the model).
STRIKE: dict[str, dict[str, float]] = {
    "heel":     {"ankle": 0.95, "knee": 1.12, "hip": 1.05, "tibia": 1.16},
    "midfoot":  {"ankle": 1.00, "knee": 1.00, "hip": 1.00, "tibia": 1.00},
    "forefoot": {"ankle": 1.18, "knee": 0.90, "hip": 0.98, "tibia": 0.86},
}

WALK_PACE_MOD: dict[str, float] = {"slow": 0.92, "normal": 1.0, "brisk": 1.10}


def speed_mod(mph: float) -> float:
    """Running GRF scales ~linearly with pace (~+8%/mph). Ref: Nilsson 1989."""
    return 0.52 + mph * 0.08


def cadence_mod(spm: float, is_walk: bool) -> float:
    """More steps/min -> shorter stride -> less drop per step. Activity-aware.

    Walking: gentle, centred on 115 spm (~0.3%/spm). Ref: Menz 2003.
    Running: ~0.6%/spm relief above 170, penalty below. Ref: Heiderscheit 2011.
    """
    if is_walk:
        d = spm - 115
        return 1 + abs(d) * 0.003 if d <= 0 else 1 - min(d, 20) * 0.003
    d = spm - 170
    return 1 + abs(d) * 0.006 if d <= 0 else 1 - min(d, 20) * 0.006


def steps_per_mile(cadence_spm: float, mph: float) -> float:
    """Geometry: stride length from pace & cadence, then steps to cover a mile."""
    ft_per_min = mph * 5280 / 60
    stride_len = ft_per_min / (cadence_spm / 2)  # feet per stride (2 steps)
    step_len = stride_len / 2                     # feet per step
    return 5280 / step_len


@dataclass(frozen=True)
class Runner:
    """A single configuration the model can be evaluated for."""
    bw_lb: float
    activity: Activity = "running"
    mph: float = 8.0
    cadence: float | None = None      # None -> activity default (115 walk / 170 run)
    strike: Strike = "midfoot"
    walk_pace: WalkPace = "normal"

    @property
    def is_walk(self) -> bool:
        return self.activity == "walking"

    @property
    def cad(self) -> float:
        if self.cadence is not None:
            return self.cadence
        return 115.0 if self.is_walk else 170.0

    def peak_force(self, joint: Joint) -> float:
        """Peak force (lb) through `joint` for one footstrike."""
        sm = WALK_PACE_MOD[self.walk_pace] if self.is_walk else speed_mod(self.mph)
        cm = cadence_mod(self.cad, self.is_walk)
        return (
            self.bw_lb
            * BASE[self.activity][joint]
            * sm
            * cm
            * STRIKE[self.strike][joint]
        )

    def steps(self, distance_mi: float) -> float:
        mph = self.mph if not self.is_walk else {"slow": 2.0, "normal": 3.0, "brisk": 4.0}[self.walk_pace]
        return steps_per_mile(self.cad, mph) * distance_mi

    def cumulative(self, joint: Joint, distance_mi: float) -> float:
        return self.peak_force(joint) * self.steps(distance_mi)


JOINTS: tuple[Joint, ...] = ("ankle", "knee", "hip", "tibia")

# Sanity check vs the numbers the live calculator shows.
assert abs(Runner(200, "running", 6).peak_force("ankle") - 1400) < 1
assert abs(Runner(200, "walking").peak_force("ankle") - 240) < 1
print("model OK — 200lb runner @6mph ankle:", round(Runner(200, "running", 6).peak_force("ankle")), "lb")


model OK — 200lb runner @6mph ankle: 1400 lb


In [2]:
# ── House visual style (matches the other For-Fun-Projects reels) ──────────
BG = "#0d0d0d"
CREAM = "#F5F2EC"
MIST = "#9A9690"
TERRA = "#C97B5A"   # ankle / "the tax" / high load
TAN = "#B5936A"     # tibia
BLUE = "#6A8FB5"    # knee
GREEN = "#5A9478"   # hip
RED = "#B85050"     # severe / heavy runner

JCOLOR: dict[str, str] = {"ankle": TERRA, "knee": BLUE, "hip": GREEN, "tibia": TAN}

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": CREAM,
    "axes.labelcolor": CREAM,
    "axes.edgecolor": "#333333",
    "xtick.color": MIST,
    "ytick.color": MIST,
    "font.family": "DejaVu Sans",
    "axes.grid": True,
    "grid.color": "#1e1e1e",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ELEPHANT_LB = 13000  # African bush elephant, used for the cumulative scale gag


def eyebrow(ax, text: str) -> None:
    ax.text(0, 1.06, text.upper(), transform=ax.transAxes, color=TERRA,
            fontsize=11, fontweight="bold", family="DejaVu Sans")


def save(fig, name: str, dpi: int = 180) -> None:
    fig.savefig(name, dpi=dpi, bbox_inches="tight", facecolor=BG)
    print("wrote", name)


def reel_fig():
    """Vertical 1080x1920 canvas for social reels."""
    fig = plt.figure(figsize=(6, 10.667), dpi=180)
    fig.patch.set_facecolor(BG)
    return fig


## Finding 1 — The bodyweight tax is perfectly linear, and inescapable

Every joint force scales *exactly* with bodyweight. A 150, 200, and 250 lb
runner all sit at 7.0×BW at the ankle — but that's 1,050 vs 1,400 vs 1,750 lb
of real force. Same form, same shoes, different physics. You cannot train the
slope away; it's the mass on the scale times a fixed leverage number.


In [3]:
# Knee and hip carry the same multiple (5.5×BW running), so they'd plot on top
# of each other — merge them into one labelled line in the sweep charts.
LINE_JOINTS: list[tuple[str, Joint, str]] = [
    ("Ankle", "ankle", TERRA),
    ("Knee / Hip", "knee", BLUE),
    ("Tibia", "tibia", TAN),
]

fig, ax = plt.subplots(figsize=(10, 5.6), dpi=180)
bw = np.linspace(100, 350, 200)
for label, j, color in LINE_JOINTS:
    forces = np.array([Runner(b, "running", 8.0).peak_force(j) for b in bw])
    ax.plot(bw, forces, color=color, lw=2.6, label=f"{label}  ({BASE['running'][j]:.1f}×BW)")

# Annotate the ankle gap at 150 vs 250
for b in (150, 250):
    f = Runner(b, "running", 8.0).peak_force("ankle")
    ax.scatter([b], [f], color=TERRA, s=40, zorder=5)
    ax.annotate(f"{f:,.0f} lb", (b, f), textcoords="offset points", xytext=(8, -4),
                color=CREAM, fontsize=10, fontweight="bold")

ax.set_title("Same pace, same shoes — the force just scales with your mass",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, "Finding 1 · The bodyweight tax  ·  running 8 mph, 170 spm")
ax.set_xlabel("Bodyweight (lb)")
ax.set_ylabel("Peak force per step (lb)")
ax.legend(facecolor="#161616", edgecolor="#333", labelcolor=CREAM, fontsize=10, loc="upper left")
save(fig, "01_bodyweight_tax.png")
plt.close(fig)


wrote 01_bodyweight_tax.png


## Finding 2 — Running vs walking isn't a step up, it's a cliff

The single biggest lever isn't pace or form — it's whether you run at all. And
the ankle is the hidden villain: 7×BW, higher than any other joint, while
everyone frets about their knees.


In [4]:
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=180)
x = np.arange(len(JOINTS))
w = 0.38
walk = [Runner(200, "walking").peak_force(j) for j in JOINTS]
run = [Runner(200, "running", 8.0).peak_force(j) for j in JOINTS]
ax.bar(x - w / 2, walk, w, color="#3a4a44", label="Walking")
ax.bar(x + w / 2, run, w, color=TERRA, label="Running (8 mph)")
for i, j in enumerate(JOINTS):
    ax.text(i + w / 2, run[i] + 25, f"{run[i]/walk[i]:.1f}×", ha="center",
            color=CREAM, fontsize=12, fontweight="bold")
ax.set_title("The jump from walking to running (200 lb runner)",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, "Finding 2 · The running cliff  ·  peak force per step")
ax.set_xticks(x)
ax.set_xticklabels([j.capitalize() for j in JOINTS])
ax.set_ylabel("Peak force per step (lb)")
ax.legend(facecolor="#161616", edgecolor="#333", labelcolor=CREAM, fontsize=10)
save(fig, "02_walk_vs_run_cliff.png")
plt.close(fig)


wrote 02_walk_vs_run_cliff.png


## Finding 3 — Over a marathon, the totals are astronomical

Per-step forces are abstract. Multiply by ~33,000 footstrikes and they become
tonnage. A 200 lb runner's ankle absorbs ~54 million lb over 26.2 miles —
roughly 4,200 elephants marching through one joint.


In [5]:
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=180)
dist = np.linspace(0, 26.2, 200)
r = Runner(200, "running", 8.0)
for label, j, color in LINE_JOINTS:
    cum = np.array([r.cumulative(j, d) / 1e6 for d in dist])
    ax.plot(dist, cum, color=color, lw=2.6, label=label)
    end = r.cumulative(j, 26.2) / 1e6
    ax.annotate(f"{end:.0f}M lb", (26.2, end), textcoords="offset points",
                xytext=(6, 0), color=color, fontsize=10, fontweight="bold", va="center")

ankle_end = r.cumulative("ankle", 26.2)
ax.set_title(f"One marathon = {ankle_end/1e6:.0f} million lb through your ankle",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, f"Finding 3 · Cumulative load  ·  ≈ {ankle_end/ELEPHANT_LB:,.0f} elephants (ankle)")
ax.set_xlabel("Distance (miles)")
ax.set_ylabel("Cumulative load (million lb)")
ax.set_xlim(0, 30)
ax.legend(facecolor="#161616", edgecolor="#333", labelcolor=CREAM, fontsize=10, loc="upper left")
save(fig, "03_cumulative_distance.png")
plt.close(fig)


wrote 03_cumulative_distance.png


## Finding 4 — You can't out-form a bodyweight problem

On the same 200 lb shin: bumping cadence 170→185 buys −9%, switching to forefoot
−14%. Real, but small. Add 30 lb of bodyweight and you're +15% — erasing both
fixes at once. Form is a rounding error next to mass and activity.


In [6]:
base = Runner(200, "running", 8.0, cadence=170, strike="midfoot").peak_force("tibia")
levers = [
    ("Cadence 170 → 185", Runner(200, "running", 8.0, cadence=185).peak_force("tibia") / base - 1),
    ("Forefoot strike",   Runner(200, "running", 8.0, strike="forefoot").peak_force("tibia") / base - 1),
    ("Heel strike",       Runner(200, "running", 8.0, strike="heel").peak_force("tibia") / base - 1),
    ("+30 lb bodyweight", Runner(230, "running", 8.0).peak_force("tibia") / base - 1),
    ("Slow down 8 → 6 mph", Runner(200, "running", 6.0).peak_force("tibia") / base - 1),
]
levers.sort(key=lambda t: t[1])
labels = [l[0] for l in levers]
vals = [l[1] * 100 for l in levers]
colors = [GREEN if v < 0 else RED for v in vals]

fig, ax = plt.subplots(figsize=(10, 5.6), dpi=180)
ax.barh(labels, vals, color=colors)
ax.axvline(0, color=MIST, lw=1)
for i, v in enumerate(vals):
    ax.text(v + (1.2 if v >= 0 else -1.2), i, f"{v:+.0f}%", va="center",
            ha="left" if v >= 0 else "right", color=CREAM, fontsize=11, fontweight="bold")
ax.set_title("Every form lever vs. the scale (200 lb runner, shin bone)",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, "Finding 4 · You can't out-form your mass")
ax.set_xlabel("Change in peak tibial load")
ax.set_xlim(min(vals) - 8, max(vals) + 8)
ax.grid(axis="y", visible=False)
save(fig, "04_cant_outform.png")
plt.close(fig)


wrote 04_cant_outform.png


## Finding 5 — Forefoot striking robs Peter to pay Paul

Going forefoot drops tibial load 14% — but raises ankle/Achilles load 18%. It's
a transfer between joints, not a net reduction. Switch too fast without calf
strength and you trade shin splints for Achilles tendinopathy.


In [7]:
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=180)
strikes: list[Strike] = ["heel", "midfoot", "forefoot"]
x = np.arange(len(strikes))
w = 0.38
ankle = [Runner(200, "running", 8.0, strike=s).peak_force("ankle") for s in strikes]
tibia = [Runner(200, "running", 8.0, strike=s).peak_force("tibia") for s in strikes]
ax.plot(x, ankle, "-o", color=TERRA, lw=2.6, ms=9, label="Ankle / Achilles")
ax.plot(x, tibia, "-o", color=TAN, lw=2.6, ms=9, label="Shin bone (tibia)")
for i in range(len(strikes)):
    ax.annotate(f"{ankle[i]:,.0f}", (i, ankle[i]), xytext=(0, 10), textcoords="offset points", ha="center", color=TERRA, fontsize=10, fontweight="bold")
    ax.annotate(f"{tibia[i]:,.0f}", (i, tibia[i]), xytext=(0, -16), textcoords="offset points", ha="center", color=TAN, fontsize=10, fontweight="bold")
ax.set_title("Forefoot lowers the shin — by loading the ankle",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, "Finding 5 · The strike trade-off  ·  200 lb, 8 mph")
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in strikes])
ax.set_ylabel("Peak force per step (lb)")
ax.legend(facecolor="#161616", edgecolor="#333", labelcolor=CREAM, fontsize=10)
save(fig, "05_strike_tradeoff.png")
plt.close(fig)


wrote 05_strike_tradeoff.png


## Finding 6 — Same pace, different body: the closer

Two runners, identical pace/cadence/shoes, over a 16-week / 640-mile block. The
210 lb shin eats **182 million lb more** than the 150 lb shin. That's not
"being out of shape" — that's the reason one got shin splints and the other
never felt a thing.


In [8]:
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=180)
weeks = np.arange(0, 17)
mi_per_week = 40
light = Runner(150, "running", 8.0)
heavy = Runner(210, "running", 8.0)
light_cum = np.array([light.cumulative("tibia", mi_per_week * wk) / 1e6 for wk in weeks])
heavy_cum = np.array([heavy.cumulative("tibia", mi_per_week * wk) / 1e6 for wk in weeks])
ax.fill_between(weeks, light_cum, heavy_cum, color=RED, alpha=0.14)
ax.plot(weeks, light_cum, color=GREEN, lw=2.8, label="150 lb runner")
ax.plot(weeks, heavy_cum, color=RED, lw=2.8, label="210 lb runner")
gap = heavy_cum[-1] - light_cum[-1]
ax.annotate(f"+{gap:.0f}M lb\non the heavier shin",
            xy=(15.6, (heavy_cum[-1] + light_cum[-1]) / 2),
            xytext=(6.2, heavy_cum[-1] * 0.92), color=RED, fontsize=13,
            fontweight="bold", va="center",
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.6))
ax.set_title("Same training block, same pace — the shin bill diverges",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, "Finding 6 · 16 weeks · 40 mi/wk · cumulative tibial load")
ax.set_xlabel("Training week")
ax.set_ylabel("Cumulative shin load (million lb)")
ax.legend(facecolor="#161616", edgecolor="#333", labelcolor=CREAM, fontsize=10, loc="upper left")
save(fig, "06_same_pace_diff_body.png")
plt.close(fig)


wrote 06_same_pace_diff_body.png


## Model validation — how these coefficients sit against measured forces

The per-joint multiples aren't invented. They were set to fall within — and
mostly below — the ranges measured in the biomechanics literature, including
in-vivo studies that instrumented real hip implants and Achilles tendons. Where
the evidence spreads, the model leans conservative.

| Joint (running) | Model | Measured range | Source |
|---|---|---|---|
| Ankle / Achilles | 7.0× | 6–12× BW | Komi 1990 (in-vivo tendon transducer) |
| Knee (patellofemoral) | 5.5× | 4.0–6.4× BW (5.2 ± 1.2) | Ho et al. 2022 meta-analysis |
| Hip (contact) | 5.5× | 5.0–5.5× BW jogging | Bergmann et al. 1993 (implant telemetry) |
| Tibia | 3.2× | see note* | Scott & Winter 1990 |

\*The tibia figure is a simplified impact/bending proxy. Measured **total**
tibial contact force reaches 10–14× bodyweight, but ~80% of that is muscle
contraction, not impact — so it is not directly comparable to the per-step
impact number reported here. Treat the shin output as "impact reaching the
bone," not total joint-contact force.


In [9]:
# Literature ranges (peak force as multiple of bodyweight, running).
LIT_RUNNING: dict[str, tuple[float, float, str]] = {
    "ankle": (6.0, 12.0, "Achilles / plantarflexor · in-vivo  (Komi 1990)"),
    "knee":  (4.0, 6.4,  "Patellofemoral · meta-analysis  (Ho 2022)"),
    "hip":   (5.0, 5.5,  "Hip contact · implant telemetry  (Bergmann 1993)"),
    "tibia": (1.8, 2.6,  "Impact/GRF portion  (Scott & Winter 1990)*"),
}

fig, ax = plt.subplots(figsize=(10, 5.2), dpi=180)
for y, j in enumerate(JOINTS):
    lo, hi, src = LIT_RUNNING[j]
    ax.plot([lo, hi], [y, y], color=JCOLOR[j], lw=9, solid_capstyle="round", alpha=0.45)
    mv = BASE["running"][j]
    ax.scatter([mv], [y], color=CREAM, s=95, zorder=5, edgecolor=JCOLOR[j], linewidth=2)
    ax.text(mv, y + 0.26, f"model {mv:.1f}×", ha="center", color=CREAM, fontsize=9, fontweight="bold")
    ax.text(12.4, y, src, va="center", color=MIST, fontsize=8.5)
ax.set_yticks(range(len(JOINTS)))
ax.set_yticklabels([j.capitalize() for j in JOINTS])
ax.set_xlabel("Peak force (× bodyweight), running")
ax.set_xlim(0, 12.2)
ax.set_ylim(-0.6, len(JOINTS) - 0.4)
ax.invert_yaxis()
ax.set_title("The model sits inside — and mostly below — measured forces",
             color=CREAM, fontsize=15, fontweight="bold", pad=30, loc="left")
eyebrow(ax, "Validation · white dot = model  ·  bar = measured in-vivo range")
ax.grid(axis="y", visible=False)
save(fig, "07_validation.png")
plt.close(fig)


wrote 07_validation.png


## References

All coefficients and modifiers trace to published work. Full findings and the
per-frame reel citations are in `README.md` and `REEL_SCRIPT.md`.

- **Bergmann et al. 1993** — *Hip joint loading during walking and running,
  measured in two patients.* J Biomech. In-vivo implant telemetry: walking
  ~2.4× BW, jogging ~5.5× BW. (OrthoLoad database.)
- **Komi 1990** — in-vivo Achilles tendon force during running, up to ~12× BW.
- **Ho et al. 2022** — *"May the force be with you"* systematic review /
  meta-analysis: patellofemoral joint force ~5.2 ± 1.2× BW running.
- **Scott & Winter 1990** — distal tibia contact force 10–14× BW running;
  ~80% muscle-driven, ~18% from ground reaction force.
- **Heiderscheit et al. 2011** — *Effects of step rate manipulation on joint
  mechanics during running.* MSSE 43(2):296–302. +10% cadence → ~34% less
  energy absorbed at the knee.
- **Nilsson & Thorstensson 1989** — ground reaction force scales ~linearly
  with running speed (~+8%/mph).
- **Daoud et al. 2012 / Kulmala et al. 2013 / Han et al. 2025** — foot-strike
  pattern shifts load between the tibia and the ankle/Achilles; forefoot lowers
  the impact transient but does not remove total load.
- **Menz et al. 2003 / Keller et al. 1996** — walking cadence and speed effects
  on ground reaction force.

Reference peak forces vary ±15–20% with anatomy, footwear, and surface. This is
a model of how the levers move, not a diagnosis. Not medical advice.
print("validation + references complete.")
